# Whisper Fine-tune — Giọng miền Trung Việt Nam (Colab Quickstart)

Notebook này chỉ là lớp vỏ chạy tuần tự 5 script trong thư mục `src/`. Toàn bộ logic thật sự nằm trong các file `.py` (dễ đọc, dễ review, dễ tái sử dụng ngoài Colab) — xem README.md để hiểu từng bước.

Kết quả tham khảo khi chạy đúng quy trình này (xem README): WER trên tập test giảm từ **38.10%** (Whisper-small gốc) xuống **17.95%** (đã fine-tune) — cải thiện tương đối 52.9%.

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

In [ ]:
REPO_URL = "https://github.com/hathu2006/whisper-central-vn.git"
!git clone $REPO_URL /content/whisper-central-vn
%cd /content/whisper-central-vn

In [ ]:
!pip install -q -r requirements.txt

## Mount Google Drive (khuyến nghị bật ngay từ đầu)

Colab free có thể ngắt phiên bất cứ lúc nào (idle timeout, hết giờ, đổi runtime type...), và `/content` là đĩa tạm — mất hết khi phiên kết thúc. Mount Drive + trỏ output vào đó để:
- Model checkpoint (bước 3) không bị mất giữa chừng, và có thể **tự resume** nếu bị ngắt (xem `src/03_finetune_whisper.py`).
- Model đã fine-tune (bước 3) vẫn còn đó để dùng lại ở bước 4/5 dù bạn quay lại sau vài ngày, không cần train lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["WHISPER_PROJECT_OUTPUTS_DIR"] = "/content/drive/MyDrive/whisper-central-vn/outputs"
# (Tuỳ chọn) cũng có thể trỏ DATA_DIR ra Drive tương tự nếu muốn giữ luôn dữ liệu đã lọc/tiền xử lý:
# os.environ["WHISPER_PROJECT_DATA_DIR"] = "/content/drive/MyDrive/whisper-central-vn/data"

## Bước 1 — Tải & lọc dữ liệu miền Trung

Khuyến nghị chạy thử với `--max_samples_per_split 50` trước để kiểm tra pipeline chạy đúng (1-2 phút), TRƯỚC KHI chạy full (có thể mất khá lâu vì phải tải qua mạng một phần lớn của bộ dữ liệu 59GB — xem giải thích chi tiết trong `src/01_load_and_filter_data.py`).

In [ ]:
# Chạy thử nhanh (khuyến nghị làm trước):
!python src/01_load_and_filter_data.py --max_samples_per_split 50

# Khi đã chắc chắn pipeline ổn, chạy full (bỏ comment dòng dưới, comment dòng trên):
# !python src/01_load_and_filter_data.py

## Bước 2 — Tiền xử lý (resample 16kHz + trích feature cho Whisper)

In [ ]:
!python src/02_preprocess_data.py

## Bước 3 — Fine-tune Whisper-small

Với dữ liệu thử (50 mẫu/split) bước này chỉ để kiểm tra code chạy được, WER sẽ không có ý nghĩa (quá ít dữ liệu). Chạy full data mới cho kết quả thật sự đáng đánh giá.

Nếu phiên bị ngắt giữa chừng: cứ chạy lại đúng cell này — script tự phát hiện checkpoint gần nhất trên Drive và tiếp tục, không train lại từ đầu (xem log dòng "Tìm thấy checkpoint cũ...").

In [ ]:
!python src/03_finetune_whisper.py

## Bước 4 — Đánh giá WER (model gốc vs. model đã fine-tune)

In [ ]:
!python src/04_evaluate_wer.py

## Bước 5 — Demo Gradio

Script tự phát hiện đang chạy trên Colab (qua biến môi trường) và tự bật link public `https://xxxxx.gradio.live` (hiệu lực 1 tuần) — không cần chỉnh gì thêm. Mở link đó, upload file audio hoặc thu âm trực tiếp để thử.

Lưu ý: demo được tối ưu cho audio ngắn (giống dữ liệu train, ≤30s/câu). Với audio dài hơn (vd. cả 1 đoạn video), Whisper vẫn xử lý được nhờ cơ chế long-form generation tích hợp sẵn, nhưng sẽ chạy lâu hơn nhiều (xử lý tuần tự từng cửa sổ 30 giây) và có thể gặp hallucination nhẹ ở các đoạn không có tiếng nói (nhạc nền, im lặng) — đây là giới hạn đã biết, không phải lỗi (xem mục "Hạn chế" trong README).

In [ ]:
!python src/05_app_gradio.py